In [2]:
%pip install pymupdf scikit-learn openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import numpy as np
import fitz

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

from openai import OpenAI

In [4]:
DOCUMENT_FOLDER = "documents"

pdf_files = [
    os.path.join(DOCUMENT_FOLDER, file)
    for file in os.listdir(DOCUMENT_FOLDER)
    if file.lower().endswith(".pdf")
]

print("PDF files found:\n")

for file in pdf_files:
    print("📄", os.path.basename(file))

PDF files found:

📄 dbms.pdf
📄 Graphics Hardware.pdf
📄 software_engineering.pdf
📄 syllabus.pdf


In [5]:
def load_pdfs(pdf_files):

    documents = []

    for pdf_path in pdf_files:

        doc = fitz.open(pdf_path)

        for page_number, page in enumerate(doc, start=1):

            text = page.get_text()

            if text.strip():

                documents.append({
                    "source": os.path.basename(pdf_path),
                    "page": page_number,
                    "text": text
                })

        doc.close()

    return documents
documents = load_pdfs(pdf_files)

print("Total pages loaded:", len(documents))

Total pages loaded: 24


In [6]:
for document in documents[:5]:

    print("=" * 60)
    print("SOURCE:", document["source"])
    print("PAGE:", document["page"])
    print(document["text"][:300])

SOURCE: dbms.pdf
PAGE: 1
DBMS – Exam Preparation Notes
1. Database Management System
•
A DBMS is software used to create, store, organize, retrieve, and manage data in databases.
•
Major DBMS functions include data definition, data manipulation, transaction management, security,
backup, and recovery.
2. Database Models
•
Th
SOURCE: Graphics Hardware.pdf
PAGE: 1
Graphics Hardware: Display Technology 
1. Graphics Hardware 
Graphics hardware is responsible for generating and processing visual data. It includes: 
a. GPU (Graphics Processing Unit) 
 
The brain of graphics hardware. 
 
Handles rendering of images, video, and animations. 
 
Modern GPUs (from N
SOURCE: Graphics Hardware.pdf
PAGE: 2
 
Better contrast, thinner displays, and faster response times. 
d. QLED (Quantum Dot LED) 
 
Enhanced LED with quantum dots for better color and brightness. 
e. MicroLED 
 
Emerging tech similar to OLED but with better durability and brightness. 
f. CRT (Cathode Ray Tube) 
 
Older technology

In [7]:
def clean_text(text):

    text = re.sub(r"\s+", " ", text)

    return text.strip()
print(clean_text(documents[0]["text"])[:500])

NameError: name 're' is not defined

In [ ]:
def create_chunks(documents, chunk_size=300, overlap=30):

    chunks = []

    for document in documents:

        text = clean_text(document["text"])

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end]

            if chunk_text.strip():

                chunks.append({
                    "text": chunk_text,
                    "source": document["source"],
                    "page": document["page"]
                })

            start += chunk_size - overlap

    return chunks
chunks = create_chunks(
    documents,
    chunk_size=300,
    overlap=30
)

print("Total chunks:", len(chunks))


In [ ]:
for i, chunk in enumerate(chunks[:5], start=1):

    print("=" * 70)

    print("CHUNK:", i)
    print("SOURCE:", chunk["source"])
    print("PAGE:", chunk["page"])

    print()

    print(chunk["text"])

In [ ]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

print("Number of chunks:", len(chunk_texts))

In [8]:
embedder = make_pipeline(

    TfidfVectorizer(
        stop_words="english",
        max_features=10000
    ),

    TruncatedSVD(
        n_components=20,
        random_state=0
    ),

    Normalizer()
)
index = embedder.fit_transform(chunk_texts)

print("Index shape:", index.shape)



NameError: name 'chunk_texts' is not defined

In [66]:
def retrieve(question, k=3):

    # Convert question into vector
    q = embedder.transform([question])[0]

    # Calculate similarity
    scores = index @ q

    # Get highest-scoring chunks
    top_indices = np.argsort(-scores)[:k]

    results = []

    for i in top_indices:

        results.append({
            "score": float(scores[i]),
            "text": chunks[i]["text"],
            "source": chunks[i]["source"],
            "page": chunks[i]["page"]
        })

    return results

results = retrieve(
    "What is normalization?",
    k=3
)

for result in results:

    print("=" * 70)

    print("Score:", round(result["score"], 4))
    print("Source:", result["source"])
    print("Page:", result["page"])

    print()

    print(result["text"])

Score: 0.9904
Source: dbms.pdf
Page: 1

tion. • A foreign key references a key in another relation and helps establish relationships between tables. • Candidate keys are attributes or attribute sets that can uniquely identify tuples. 4. Normalization • Normalization organizes relations to reduce redundancy and update anomalies. • First No
Score: 0.9746
Source: dbms.pdf
Page: 1

overy. 2. Database Models • The relational model organizes data into tables consisting of rows and columns. • Other common models include hierarchical, network, object-oriented, and document-oriented models. 3. Keys • A primary key uniquely identifies a record in a relation. • A foreign key referenc
Score: 0.9403
Source: syllabus.pdf
Page: 1

Demo Exam Preparation Syllabus Unit 1 – Database Management Systems • Introduction to DBMS and database concepts • Database models and relational concepts • Keys, constraints, and relationships • SQL and relational operations • Functional dependencies and normalization • Tr

In [67]:
retrieve("What is software testing?")

[{'score': 0.9792912155193203,
  'text': 'rformance, security, and usability. 5. Software Design • Software design converts requirements into an architecture and detailed solution structure. • Important design concepts include modularity, abstraction, cohesion, and coupling. 6. Software Testing • Unit testing tests individual components. • ',
  'source': 'software_engineering.pdf',
  'page': 1},
 {'score': 0.9788047241043735,
  'text': '• Transactions, ACID properties, and concurrency control Unit 2 – Software Engineering • Introduction to software engineering • Software development life cycle • Software process models • Requirements engineering • Software architecture and design • Software testing • Software maintenance Unit 3 – C',
  'source': 'syllabus.pdf',
  'page': 1},
 {'score': 0.9488727063106959,
  'text': 'Software Engineering – Exam Preparation Notes 1. Introduction • Software engineering is the systematic and disciplined approach to the development, operation, maintenance, 

In [68]:
def build_context(question, k=3):

    results = retrieve(question, k)

    context_parts = []

    for result in results:

        context_parts.append(
            f"""
SOURCE: {result['source']}
PAGE: {result['page']}

{result['text']}
"""
        )

    context = "\n".join(context_parts)

    return context
context = build_context(
    "What is normalization?",
    k=3
)

print(context)


SOURCE: dbms.pdf
PAGE: 1

tion. • A foreign key references a key in another relation and helps establish relationships between tables. • Candidate keys are attributes or attribute sets that can uniquely identify tuples. 4. Normalization • Normalization organizes relations to reduce redundancy and update anomalies. • First No


SOURCE: dbms.pdf
PAGE: 1

overy. 2. Database Models • The relational model organizes data into tables consisting of rows and columns. • Other common models include hierarchical, network, object-oriented, and document-oriented models. 3. Keys • A primary key uniquely identifies a record in a relation. • A foreign key referenc


SOURCE: syllabus.pdf
PAGE: 1

Demo Exam Preparation Syllabus Unit 1 – Database Management Systems • Introduction to DBMS and database concepts • Database models and relational concepts • Keys, constraints, and relationships • SQL and relational operations • Functional dependencies and normalization • Transactions, ACID propertie



In [69]:
import os
from openai import OpenAI

os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6JFRh9BIwn3YI4EcLO_3t65bAaC8Rb0tEgGB4ViSBx1Ag"

client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

print("Gemini client ready!")

Gemini client ready!


In [70]:
def generate_answer(question, context):

    prompt = f"""
You are an exam preparation assistant.

Answer the student's question using ONLY the study material
provided below.

Do not use outside knowledge.

If the study material does not contain enough information,
say:

"I couldn't find enough information in the provided study material."

STUDY MATERIAL
===============
{context}
===============

QUESTION
========
{question}

Instructions:

- Give a clear answer.
- Use simple language.
- Keep important technical terminology.
- Organize the answer with headings or bullet points when useful.
- Do not invent information.
"""

    response = client.chat.completions.create(
        model="gemini-3.5-flash-lite",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful exam preparation assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [71]:
ask_rag("What is normalization?")

QUESTION
What is normalization?

ANSWER
Based on the provided study material, here is the answer regarding normalization:

* **Definition:** Normalization organizes relations to reduce redundancy and update anomalies.

SOURCES
📄 dbms.pdf | Page 1 | Score 0.9904
📄 dbms.pdf | Page 1 | Score 0.9746
📄 syllabus.pdf | Page 1 | Score 0.9403


'Based on the provided study material, here is the answer regarding normalization:\n\n* **Definition:** Normalization organizes relations to reduce redundancy and update anomalies.'

In [1]:
def exam_rag(question, mode="explain", k=3):

    # ==========================================
    # 1. RETRIEVE RELEVANT CHUNKS
    # ==========================================

    results = retrieve(question, k)

    # ==========================================
    # 2. BUILD CONTEXT
    # ==========================================

    context_parts = []

    for result in results:

        context_parts.append(
            f"""
SOURCE: {result['source']}
PAGE: {result['page']}

{result['text']}
"""
        )

    context = "\n".join(context_parts)

    # ==========================================
    # 3. SELECT EXAM MODE
    # ==========================================

    if mode == "explain":

        instruction = """
Explain the topic clearly for a student.
Use simple language while preserving important
technical terminology.
"""

    elif mode == "short":

        instruction = """
Give a concise exam answer.
Focus only on the most important points.
"""

    elif mode == "5-mark":

        instruction = """
Write a well-structured 5-mark exam answer.

Use this structure when the material supports it:

1. Definition
2. Explanation
3. Important points
4. Example

Do not invent an example if one is not present
in the study material.
"""

    elif mode == "mcq":

        instruction = """
Create 5 multiple-choice questions from the
provided study material.

Each question must have:

A.
B.
C.
D.

Then provide the correct answer.

Do not use information outside the study material.
"""

    else:

        instruction = """
Answer the question using only the provided
study material.
"""

    # ==========================================
    # 4. CREATE PROMPT
    # ==========================================

    prompt = f"""
You are an Exam Preparation RAG assistant.

IMPORTANT:
Use ONLY the supplied study material.

{instruction}

If the answer cannot be found in the supplied
study material, clearly say so.

STUDY MATERIAL
====================
{context}
====================

QUESTION
====================
{question}
"""

    # ==========================================
    # 5. CALL GEMINI
    # ==========================================

    response = client.chat.completions.create(
        model="gemini-3.5-flash-lite",
        messages=[
            {
                "role": "system",
                "content": "You are an exam preparation assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    # ==========================================
    # 6. DISPLAY ANSWER
    # ==========================================

    print("=" * 70)
    print("QUESTION")
    print("=" * 70)

    print(question)

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)

    print(answer)

    # ==========================================
    # 7. DISPLAY SOURCES
    # ==========================================

    print("\n" + "=" * 70)
    print("SOURCES")
    print("=" * 70)

    for result in results:

        print(
            f"📄 {result['source']} "
            f"| Page {result['page']} "
            f"| Score {result['score']:.4f}"
        )

    return answer

response = client.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=[
        {
            "role": "user",
            "content": "Explain DBMS in two sentences."
        }
    ]
)

print(response.choices[0].message.content)

NameError: name 'client' is not defined

In [ ]:
exam_rag(
    "What is normalization?",
    mode="explain"
)

In [ ]:
exam_rag(
    "What is a primary key?",
    mode="short"
)

In [ ]:
exam_rag(
    "Explain normalization.",
    mode="5-mark"
)

In [ ]:
exam_rag(
    "Normalization",
    mode="mcq"
)

In [ ]:
while True:

    print("\n" + "=" * 70)
    print("📚 EXAM PREPARATION RAG")
    print("=" * 70)

    question = input(
        "\nEnter your question (or type 'exit'): "
    ).strip()

    if question.lower() == "exit":
        print("\nGoodbye! 👋")
        break

    question_lower = question.lower()

    if "mcq" in question_lower or "multiple choice" in question_lower:
        mode = "mcq"

    elif "5 mark" in question_lower or "5-mark" in question_lower:
        mode = "5-mark"

    elif "short answer" in question_lower or "in short" in question_lower:
        mode = "short"

    else:
        mode = "explain"

    print("\n🎯 Detected mode:", mode)

    exam_rag(
        question,
        mode=mode,
        k=3
    )